In [2]:
import numpy as np
import pandas as pd
import fastf1
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [3]:
# Loading race data from fastf1 api

race = fastf1.get_session(2026, "Barcelona", "R")
race.load()
laps = race.laps

req         WARNING 	DEFAULT CACHE ENABLED! (103.85 MB) C:\Users\charl\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Barcelona Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['44

In [4]:
# Sort data to use to generate model
laps1 = laps[["Team", "Driver", "LapTime", "LapNumber", "PitOutTime", "PitInTime", "Compound", "TyreLife", "TrackStatus", "IsAccurate"]]
laps2 = laps1[(laps1["LapNumber"] == 1.0) | ((laps1["IsAccurate"] == True) & (laps1["TrackStatus"] == "1"))]
laps3 = laps2[laps2["Team"].isin(["McLaren", "Mercedes", "Ferrari", "Red Bull Racing"])]
laps3 = laps3[laps3["Driver"] != "HAD"]
lapsA = laps3[laps3["PitInTime"].isna() & laps3["PitOutTime"].isna()]
lapsA["LapTime"] = lapsA["LapTime"].dt.total_seconds()
compound_dummies = pd.get_dummies(lapsA["Compound"], prefix="Compound")
lapsfinal = pd.concat([lapsA, compound_dummies], axis=1)
lapsfinal

,Team,Driver,LapTime,LapNumber,PitOutTime,PitInTime,Compound,TyreLife,TrackStatus,IsAccurate,Compound_HARD,Compound_MEDIUM,Compound_SOFT
0,McLaren,NOR,87.198,1.0,NaT,NaT,MEDIUM,1.0,1,False,False,True,False
1,McLaren,NOR,82.251,2.0,NaT,NaT,MEDIUM,2.0,1,True,False,True,False
2,McLaren,NOR,82.422,3.0,NaT,NaT,MEDIUM,3.0,1,True,False,True,False
3,McLaren,NOR,82.773,4.0,NaT,NaT,MEDIUM,4.0,1,True,False,True,False
4,McLaren,NOR,83.287,5.0,NaT,NaT,MEDIUM,5.0,1,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1168,McLaren,PIA,83.228,59.0,NaT,NaT,HARD,23.0,1,True,True,False,False
1169,McLaren,PIA,82.546,60.0,NaT,NaT,HARD,24.0,1,True,True,False,False
1170,McLaren,PIA,82.671,61.0,NaT,NaT,HARD,25.0,1,True,True,False,False
1174,McLaren,PIA,83.332,65.0,NaT,NaT,HARD,29.0,1,True,True,False,False


In [5]:
# Generate lap time prediction model using random forest regression

# Define features and target

model_features = ["LapNumber", "TyreLife", "Compound_HARD", "Compound_MEDIUM", "Compound_SOFT"]
X = lapsfinal[model_features].values
y = lapsfinal["LapTime"].values

# define model sets and parameters

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 5)

model = RandomForestRegressor(
    n_estimators=400,
    max_depth=30,
    min_samples_split=3,
    min_samples_leaf=2,
    random_state=5
)


In [6]:
# Fitting model 

model.fit(X_train, y_train)

,n_estimators,400
,criterion,'squared_error'
,max_depth,30
,min_samples_split,3
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
# Testing model and calculating residual standard deviation to add noise

y_pred = model.predict(X_test)
print(mean_absolute_error(y_test, y_pred))
print(r2_score(y_test, y_pred))
residuals = y_test - y_pred

laptime_std = residuals.std()

0.3596218830469732
0.8274781461483005


In [8]:
# Setting core parameters 

n_laps = 66
n_simulations = 1

strategies = [
    [[-1, "HARD"], [41, "MEDIUM"]],
    [[-1, "MEDIUM"], [20, "MEDIUM"], [39, "HARD"]],
    [[-1, "SOFT"], [12, "MEDIUM"], [28, "MEDIUM"], [44, "HARD"]],
    [[-1, "SOFT"], [12, "HARD"], [27, "MEDIUM"], [43, "HARD"]],
    [[-1, "MEDIUM"], [13, "HARD"], [37, "HARD"]]
]



tyre_compound_encoding = {"SOFT": [0, 0, 1],
                 "MEDIUM": [0, 1, 0],
                 "HARD": [1, 0, 0]}

flag_states = {0: "G",
              1: "SC",
              2: "VSC"}

flag_transition_matrix = np.array([[0.963, 0.03, 0.007],
                            [0.55, 0.45, 0],
                            [0.9, 0, 0.1]])



n_strategies = len(strategies)

lap_times = np.empty([n_strategies, n_simulations, n_laps], dtype = float)

In [9]:
# Generate flags for n_simulations and n_laps

def get_all_flags(n_laps, n_simulations, flag_states, flag_transition_matrix):
    race_flags = np.empty([n_simulations, n_laps], dtype = object)
    race_flags[:, 0] = flag_states[0]
    for i in range(n_simulations):
        current_state = 0
        for j in range(1, n_laps):
            current_state = np.random.choice([0,1,2], p = flag_transition_matrix[current_state])
            race_flags[i][j] = flag_states[current_state]

    return race_flags

In [10]:
# Generating lap times

# tyre cliff simualtion 

def tyre_cliff(tyre_life, current_tyre_compound):
    if current_tyre_compound == "SOFT" and tyre_life > 15:
        return (0.15 * (tyre_life - 15)) ** 1.5
        
    if current_tyre_compound == "MEDIUM" and tyre_life > 21:
        return (0.12 * (tyre_life - 21)) ** 1.3
        
    if current_tyre_compound == "HARD" and tyre_life > 25:
        return (0.12 * (tyre_life - 25)) ** 1.2
        
    return 0


# non pit lap 

def race_lap_time(current_flag, prediction_data):
    tyre_life = prediction_data[0][1]
    
    if current_flag == "G":
        laptime = float(model.predict(prediction_data)[0]) + np.random.normal(0, laptime_std) + tyre_cliff(tyre_life, current_tyre_compound)
        tyre_life = prediction_data[0][1] + 1
        return laptime, tyre_life 
    
    elif current_flag == "SC":
        laptime = float((1.56 * model.predict(prediction_data))[0]) + np.random.normal(0, laptime_std)
        tyre_life = prediction_data[0][1] + 0.5
        return laptime, tyre_life
   
    elif current_flag == "VSC":
        laptime = float((1.33 * model.predict(prediction_data))[0]) + np.random.normal(0, laptime_std)
        tyre_life = prediction_data[0][1] + 0.5
        return  laptime, tyre_life

# pit lap

def pit_lap(current_flag, prediction_data):
    tyre_life = prediction_data[0][1]
    if current_flag == "G":
        return  float(model.predict(prediction_data)[0]) + np.random.normal(0, laptime_std) + 22 + np.random.normal(0, 0.7)+tyre_cliff(tyre_life, current_tyre_compound)
    
    elif current_flag == "SC":
        return  float(1.56 * model.predict(prediction_data)[0]) + np.random.normal(0, laptime_std) + 16.3 + np.random.normal(0, 0.7)
   
    elif current_flag == "VSC":
        return  float(1.33 * model.predict(prediction_data)[0]) + np.random.normal(0, laptime_std) + 13.5 + np.random.normal(0, 0.7)

In [11]:
race_flags = get_all_flags(n_laps, n_simulations, flag_states, flag_transition_matrix)

for strategy in range(n_strategies):

    
    for simulation in range(n_simulations):
        tyre_life = 1
        current_tyre_compound = strategies[strategy][0][1]
        stint = 1
        
        for lap in range(n_laps):
            current_flag = race_flags[simulation, lap]
            prediction_data = [[lap+1, tyre_life] + tyre_compound_encoding[current_tyre_compound]]

            if stint < len(strategies[strategy]):
            
                if lap == strategies[strategy][stint][0]:

                    lap_times[strategy, simulation, lap] = pit_lap(current_flag, prediction_data)
                    current_tyre_compound = strategies[strategy][stint][1]
                    stint += 1
                    tyre_life = 1
            
                else:
                    lap_times[strategy, simulation, lap], tyre_life = race_lap_time(current_flag, prediction_data)
            
            else:
                lap_times[strategy, simulation, lap], tyre_life = race_lap_time(current_flag, prediction_data)
                
            
            
print("finished")          

finished


In [16]:
# Calculating mean race time for each strategy

race_times = np.mean(np.sum(lap_times, axis = 2), axis  = 1)

for _ in range(n_strategies):
    print(f"Strategy {_ + 1} average race time: {race_times[_]:.2f}")
    

Strategy 1 average race time: 5597.38
Strategy 2 average race time: 5578.18
Strategy 3 average race time: 5587.92
Strategy 4 average race time: 5573.94
Strategy 5 average race time: 5595.28
